# RV-ANDROID playground

# Config

## Local


Requisites

`sudo apt install python3.12-dev nvidia-cuda-toolkit bitsandbytes triton`

```
nvidia-smi
nvcc --version
```

In [ ]:
# Check GPU

import torch
if torch.cuda.is_available():
    print("GPU is available")
else:
    print("GPU is not available")

In [1]:
# Log in HF

import os
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv(override=True)

hf_token = os.getenv('HF_TOKEN')

login(hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Google colab

In [ ]:
!pip install -q gradio diffusers transformers accelerate torch Pillow python-dotenv torchvision
!pip install -q -U bitsandbytes

#datasets

In [ ]:
# Clone RVSec
from google.colab import userdata, drive

!rm -Rf sample_data/

#https://github.com/ad17171717/YouTube-Tutorials/blob/main/Google%20Colab%20Tutorials/Google_Colab_%2B_Git_Pushing_Changes_to_a_GitHub_Repo!.ipynb
!git config --global user.name "phtcosta"
!git config --global user.email "phtcosta@gmail.com"

# https://github.com/settings/tokens
github_token = userdata.get('GITHUB_TOKEN')
!git clone --branch develop https://{github_token}@github.com/PAMunb/rvsec.git

%cd rvsec/rv-android/
!pip install -q -r requirements.txt

In [ ]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Log in HF
from huggingface_hub import login
hf_token = userdata.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

In [ ]:
!git status
!pwd

In [ ]:
# drive.flush_and_unmount()
# !git add --all
# !git commit -a -m "Just testing"
# !git remote -v

#  Experiments

In [2]:
# Imports

from IPython.display import Markdown, display, update_display #, Image
import gradio as gr
from PIL import Image
import numpy as np
import os
import glob
import json
from typing import List
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig, AutoModelForSeq2SeqLM
import torch
from transformers import pipeline, AutoModelForSpeechSeq2Seq, AutoProcessor, WhisperConfig, WhisperForConditionalGeneration

from rvandroid.llm.huggingface import HuggingFaceLLM
from rvandroid.llm.llm import LanguageModel
from rvandroid.llm.ollama import OllamaLLM
import rvandroid.parser.droidbot_state_parser as state_parser

In [3]:
# Constants

LLM_OLLAMA = "ollama"
LLM_HF = "hf"


In [5]:
# DEFAULT_LLM_TYPE = LLM_HF
# DEFAULT_MODEL = HuggingFaceLLM.LLAMA
# MODELS = HuggingFaceLLM.MODELS

DEFAULT_LLM_TYPE = LLM_OLLAMA
DEFAULT_MODEL = OllamaLLM.LLAMA
MODELS = OllamaLLM.MODELS


def create_llm(model_name: str = DEFAULT_MODEL, type: str = DEFAULT_LLM_TYPE) -> LanguageModel:
    if type == LLM_OLLAMA:
        return OllamaLLM(model_name)
    return HuggingFaceLLM(model_name)


In [6]:
def read_text_file(file_path):
    with open(file_path, 'r') as file:
        text = file.read()
    return text

def read_files_by_extension(folder: str, extension: str = "*.gesda"):
    files = glob.glob(os.path.join(folder, extension))
    for file in files:
      text = read_text_file(file)
      yield file, text
      

## Static Analysis

In [7]:
static_folder = "/home/pedro/desenvolvimento/RV_ANDROID/teste_llm/static"
# static_folder = "/content/drive/MyDrive/llms/rvandroid/static"


def create_messages(system_msg: str, prompt: str, json_text: str) -> list[dict[str, str]]:
    messages=[
        {"role": "system", "content": system_msg },
        {"role": "user", "content": prompt.format(json_text)}
    ]
    return messages


### GESDA

In [8]:
base_gesda_system_msg = """You are an expert assistant in testing the interface of Android applications, and you use this knowledge to make useful summaries about the components (activities, windows, widgets) contained on the screen. Some widgets may have information about which method will be called when it is clicked, others may have information about the assignment of this widget to a field declared in the class, listing all those that are relevant in the context of interface testing, and suggesting the possible actions on this component (click, set text, select item). The information about the application that must be understood is contained in a string in json format, which will be passed to you.
"""
base_gesda_prompt = "Make a summary of the application 'cryptoapp' which has the following information in json format: {}"



In [9]:
# Basic example (GESDA)

text = read_text_file(static_folder+"/cryptoapp.apk.gesda")
print(text)
messages=create_messages(base_gesda_system_msg, base_gesda_prompt, text)
print(messages)

print("Generating ...")
llm = create_llm()
response_gesda = llm.generate(messages)
# print(response_gesda)
display(Markdown(response_gesda))
llm.clean()

del llm

{"fileName":"cryptoapp.apk","packageName":"br.unb.cic.cryptoapp","windows":[{"id":1,"name":"br.unb.cic.cryptoapp.cipher.CipherActivity","isMain":false,"layoutFileName":"activity_cipher","type":"ACT","widgets":[{"widgetId":"2131296709","type":"TEXT_VIEW","name":"textViewCypherEncryptResult","field":"\u003cbr.unb.cic.cryptoapp.cipher.CipherActivity: android.widget.TextView textViewEncryptResult\u003e"},{"widgetId":"2131296354","type":"BUTTON","name":"btn_cipher_encrypt","field":"\u003cbr.unb.cic.cryptoapp.cipher.CipherActivity: android.widget.Button btnEncrypt\u003e","listeners":[{"type":"OnClickListener","callbackMethod":{"name":"onClick","className":"br.unb.cic.cryptoapp.cipher.CipherActivity$1","signature":"\u003cbr.unb.cic.cryptoapp.cipher.CipherActivity$1: void onClick(android.view.View)\u003e","modifiers":1},"registeredInFile":false}]},{"widgetId":"2131296437","type":"EDIT_TEXT","name":"editTextCipherEncrypt","field":"\u003cbr.unb.cic.cryptoapp.cipher.CipherActivity: android.widget

Based on the provided JSON information, here is a summary of the 'cryptoapp' Android application:

**Activities**

* `br.unb.cic.cryptoapp.cipher.CipherActivity`: The main activity for encryption, containing:
	+ A text view to display encryption results
	+ A button to trigger encryption
	+ An edit text field for inputting text to be encrypted
* `br.unb.cic.cryptoapp.MainActivity`: The main entry point of the application, containing:
	+ Two buttons: "Cipher" and "Message Digest"
	+ An options menu with one item "Home"
* `br.unb.cic.cryptoapp.messagedigest.MessageDigestActivity`: An activity for message digest calculations, containing:
	+ A text view to display results
	+ A spinner to select the algorithm
	+ An edit text field for inputting text to be hashed
	+ A button to generate the hash

**Widgets**

* `textViewCypherEncryptResult` (text view): Displays encryption results in `br.unb.cic.cryptoapp.cipher.CipherActivity`
* `btn_cipher_encrypt` (button): Triggers encryption in `br.unb.cic.cryptoapp.cipher.CipherActivity`
* `editTextCipherEncrypt` (edit text field): Input field for encrypting text in `br.unb.cic.cryptoapp.cipher.CipherActivity`
* `cipherTextView` (text view): Not specified, but possibly related to encryption results
* `buttonCipher` (button): Triggers showing the Cipher activity
* `buttonMessageDigest` (button): Triggers showing the Message Digest activity
* `spinnerAlgorithm` (spinner): Selects the algorithm for message digest calculations in `br.unb.cic.cryptoapp.messagedigest.MessageDigestActivity`
* `editTextMessageDigest` (edit text field): Input field for hashing text in `br.unb.cic.cryptoapp.messagedigest.MessageDigestActivity`
* `textViewMessageDigestResult` (text view): Displays results of message digest calculations in `br.unb.cic.cryptoapp.messagedigest.MessageDigestActivity`
* `buttonGenerateHash` (button): Generates the hash in `br.unb.cic.cryptoapp.messagedigest.MessageDigestActivity`

**Possible Actions**

* Click on:
	+ `btn_cipher_encrypt`: Triggers encryption in `br.unb.cic.cryptoapp.cipher.CipherActivity`
	+ `buttonCipher`: Triggers showing the Cipher activity
	+ `buttonMessageDigest`: Triggers showing the Message Digest activity
	+ `spinnerAlgorithm` items: Selects an algorithm for message digest calculations in `br.unb.cic.cryptoapp.messagedigest.MessageDigestActivity`
* Set text:
	+ In `editTextCipherEncrypt`: Inputs text to be encrypted in `br.unb.cic.cryptoapp.cipher.CipherActivity`
	+ In `editTextMessageDigest`: Inputs text to be hashed in `br.unb.cic.cryptoapp.messagedigest.MessageDigestActivity`

Note: The JSON information only provides a limited view of the application's functionality. Further testing and inspection may reveal additional features or behaviors not included here.

In [ ]:
# Basic Example - LOCAL (parser)

text = read_text_file(static_folder+"/cryptoapp.apk.gesda")

# fazer o parse do arquivo .gesda (json) para dictionary
def read_gesda_file(file_path: str):
    try:
        with open(file_path, 'r') as file:
            data = json.load(file)
        return data
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return None
    except json.JSONDecodeError:
        print(f"Failed to parse JSON from file: {file_path}")
        return None
    
gesda = read_gesda_file(static_folder+"/cryptoapp.apk.gesda")
print(gesda)

def parse_gesda_file(gesda_data):
    for window in gesda_data["windows"]:
        print(window["name"])
            
            

parse_gesda_file(gesda)


In [ ]:
# Gradio

llm = create_llm()

# Get GESDA files (path) and texts (contents)
files, texts = get_gesda_files(static_folder)

current_index = 0  # Index of the currently displayed file
selected_model = DEFAULT_MODEL # Currently selected model

def get_system_prompt():
   return base_gesda_system_msg

def get_user_prompt():
   return base_gesda_prompt

def generate_output(system_prompt, user_prompt):
  messages = create_messages(system_prompt, user_prompt, texts[current_index])
  response = llm.generate(messages)
  return response

def display_file(index):
    """Returns the filename at the given index."""
    return files[index]

def process_selection(selected_option):
  """Processes the model selection."""
  global selected_model
  selected_model = selected_option
  print(f"You selected: {selected_option}")
  global llm # Make llm global so it can be reassigned
  llm = create_llm(selected_model, DEFAULT_LLM_TYPE) # Initialize the LLM with the selected model

def advance_file():
    """Advances to the next file in the list."""
    global current_index
    current_index = (current_index + 1) % len(files)  # Wraps around to the beginning if at the end
    return display_file(current_index)

def go_back_file():
    """Goes back to the previous file in the list."""
    global current_index
    current_index = (current_index - 1) % len(files)  # Wraps around to the end if at the beginning
    return display_file(current_index)

def reset(system_prompt, user_prompt, result_text):
  """Resets the system and user prompts and the result text."""
  return base_gesda_system_msg, base_gesda_prompt, ""

def clear_memory():
  """Clears the LLM memory and CUDA cache."""
  if llm is not None:
    llm.clean()
  torch.cuda.empty_cache()
  return ""

with gr.Blocks() as demo:
    with gr.Row():
      filename = gr.Textbox(label="GESDA file", lines=1, value=display_file(current_index))

    with gr.Row():
      previous_button = gr.Button("Previous")
      next_button = gr.Button("Next")

    with gr.Row():
      system_textbox = gr.Textbox(label="System Prompt", value=get_system_prompt()) #, lines=5)
      prompt_textbox = gr.Textbox(label="User Prompt", value=get_user_prompt()) #, lines=3)

    with gr.Row():
      model_dropdown = gr.Dropdown(
        label="Select MODEL",
        choices=llm.models(),
        value=DEFAULT_MODEL
      )
      with gr.Row():
        generate_button = gr.Button("Generate")
        reset_button = gr.Button("Reset")

    with gr.Row():
      result_textbox = gr.Textbox(lines=10)

    with gr.Row():
      clear_button = gr.Button("Clear memory")

    previous_button.click(go_back_file, outputs=filename)
    next_button.click(advance_file, outputs=filename)
    model_dropdown.change(fn=process_selection, inputs=model_dropdown)
    generate_button.click(generate_output, inputs=[system_textbox, prompt_textbox], outputs=result_textbox)
    reset_button.click(reset, inputs=[system_textbox, prompt_textbox, result_textbox], outputs=[system_textbox, prompt_textbox, result_textbox])
    clear_button.click(clear_memory)

demo.launch(debug=True)

In [ ]:
# Clean up

if llm is not None:
    llm.clean()
    
del llm
del demo

### GATOR

In [13]:
base_gator_system_msg = """You are an expert in testing the interface of Android applications, and you use this knowledge to make useful summaries about the components (activities, windows, widgets and transitions between windows) contained on the screen. Some widgets may have information about which method will be called when clicked, others may have information about the assignment of this widget to a field declared in the class. List all those that are relevant in the context of interface testing, and suggest the possible actions on this component (click, set text, select item). The main objective is to increase the test coverage according to the window transition graph to try to cover all the activities.
There are two basic sections in the json: one with information about the application windows and another with information about the transitions between windows, including the events that cause this transition. The information about the windows is in the following format "{"id":1349,"name":"br.unb.cic.cryptoapp.MainActivity"}", where the activity br.unb.cic.cryptoapp.MainActivity has the identifier 1349. Another example: "{ "id": 1336, "name": "br.unb.cic.cryptoapp.messagedigest.MessageDigestActivity" }, where the activity br.unb.cic.cryptoapp.messagedigest.MessageDigestActivity has the identifier 1336"
The information about the transitions between the windows is in the following format:
{
"sourceId": 1349,
"getTargetId": 1336,
"events": [
{
"type": "click",
"handler": "\u003cbr.unb.cic.cryptoapp.MainActivity: void showScreenMessageDigest(android.view.View)\u003e",
"widgetId": 2131296357,
"widgetClass": "android.widget.Button",
"widgetName": "buttonMessageDigest"
}
]
}
The above example shows a transition between the activities (windows) br.unb.cic.cryptoapp.MainActivity (with id 1349) and br.unb.cic.cryptoapp.messagedigest.MessageDigestActivity (with id 1336). This transition is activated when the button 'buttonMessageDigest' is clicked. When the button is clicked, the event is passed to the method "\u003cbr.unb.cic.cryptoapp.MainActivity: void showScreenMessageDigest(android.view.View)\u003e", which will start the intent corresponding to MessageDigestActivity
"""
base_gator_prompt = "Create a document containing information about the 'cryptoapp' application. The response should contain the list of all windows and a mapping of the transitions between them, including the event that caused the transition between screens. The text containing the application data: {}"

In [14]:
# Basic example (GATOR)

text = read_text_file(static_folder+"/cryptoapp.apk.wtg")
print(text)
# messages=create_messages(base_gesda_system_msg, base_gator_prompt, text)
messages=create_messages(base_gator_system_msg, base_gator_prompt, text)
print(messages)

print("Generating ...")
llm = create_llm()
response_gator = llm.generate(messages)
display(Markdown(response_gator))
llm.clean()

del llm

{"windows":[{"id":1533,"name":"presto.android.gui.stubs.PrestoFakeLauncherNodeClass"},{"id":1349,"name":"br.unb.cic.cryptoapp.MainActivity"},{"id":1336,"name":"br.unb.cic.cryptoapp.messagedigest.MessageDigestActivity"},{"id":1342,"name":"android.view.Menu"},{"id":1339,"name":"br.unb.cic.cryptoapp.cipher.CipherActivity"}],"transitions":[{"sourceId":1339,"getTargetId":1339,"events":[{"type":"implicit_home_event","handler":"","widgetId":1339,"widgetClass":"br.unb.cic.cryptoapp.cipher.CipherActivity"}],"callbacks":[]},{"sourceId":1342,"getTargetId":1349,"events":[{"type":"implicit_back_event","handler":"","widgetId":1342,"widgetClass":"android.view.Menu"}],"callbacks":[]},{"sourceId":1339,"getTargetId":1339,"events":[{"type":"implicit_rotate_event","handler":"","widgetId":1339,"widgetClass":"br.unb.cic.cryptoapp.cipher.CipherActivity"}],"callbacks":[{"type":"implicit_lifecycle_event","handler":"\u003cbr.unb.cic.cryptoapp.cipher.CipherActivity: void onCreate(android.os.Bundle)\u003e","widge

This appears to be a list of events and handlers for an Android application. The format is not human-readable, but I can try to break it down into smaller sections.

**Events**

1. `presto.android.gui.stubs.PrestoFakeLauncherNodeClass` - A launch event
2. `br.unb.cic.cryptoapp.MainActivity.onCreate(android.os.Bundle)` - An activity creation event
3. `br.unb.cic.cryptoapp.messagedigest.MessageDigestActivity.onCreate(android.os.Bundle)` - An activity creation event for the MessageDigest Activity
4. `br.unb.cic.cryptoapp.MainActivity.showScreenMessageDigest(android.view.View)` - A button click event to show the MessageDigest screen
5. `br.unb.cic.cryptoapp.MainActivity.showScreenCipher(android.view.View)` - A button click event to show the Cipher screen
6. `br.unb.cic.cryptoapp(MainActivity.boolean onCreateOptionsMenu(android.view.Menu))` - An event for setting up the menu

**Handlers**

1. `presto.android.gui.stubs.PrestoFakeLauncherNodeClass.onCreate(android.os.Bundle)`
2. `br.unb.cic.cryptoapp.MainActivity.onCreate(android.os.Bundle)`
3. `br.unb.cic.cryptoapp.messagedigest.MessageDigestActivity.onCreate(android.os.Bundle)`
4. `br.unb.cic.cryptoapp.MainActivity.showScreenMessageDigest(android.view.View)`
5. `br.unb.cic.cryptoapp.MainActivity.showScreenCipher(android.view.View)`
6. `br.unb.cic.cryptoapp(MainActivity.boolean onCreateOptionsMenu(android.view.Menu))`

**Menu**

1. `br.unb.cic.cryptoapp.MainActivity.onCreateOptionsMenu(android.view.Menu)`

The menu event is handled by the `MainActivity` class, which creates a menu for the application.

Note: The code is not readable in its current format, and it's likely that this is a dump of an Android app's debug logs.

### Gradio

In [ ]:
def get_files_by_extension(folder: str, extension = "*.gesda"):
    filenames = []
    texts = []
    for file, text in read_files_by_extension(folder, extension):
      filenames.append(file)
      texts.append(text)
    return filenames, texts


# GESDA
# files, texts = get_files_by_extension(static_folder, "*.gesda")
# gradio_system_prompt = base_gesda_system_msg
# gradio_user_prompt = base_gesda_prompt

# GATOR
files, texts = get_files_by_extension(static_folder, "*.wtg")
gradio_system_prompt = base_gator_system_msg
gradio_user_prompt = base_gator_prompt



llm = create_llm()

current_index = 0  # Index of the currently displayed file
selected_model = DEFAULT_MODEL # Currently selected model

def get_system_prompt():
   return gradio_system_prompt

def get_user_prompt():
   return gradio_user_prompt

def generate_output(system_prompt, user_prompt):
  messages = create_messages(system_prompt, user_prompt, texts[current_index])
  response = llm.generate(messages)
  return response

def display_file(index):
    """Returns the filename at the given index."""
    return files[index]

def process_selection(selected_option):
  """Processes the model selection."""
  global selected_model
  selected_model = selected_option
  print(f"You selected: {selected_option}")
  global llm # Make llm global so it can be reassigned
  llm = create_llm(selected_model, DEFAULT_LLM_TYPE) # Initialize the LLM with the selected model

def advance_file():
    """Advances to the next file in the list."""
    global current_index
    current_index = (current_index + 1) % len(files)  # Wraps around to the beginning if at the end
    return display_file(current_index)

def go_back_file():
    """Goes back to the previous file in the list."""
    global current_index
    current_index = (current_index - 1) % len(files)  # Wraps around to the end if at the beginning
    return display_file(current_index)

def reset(system_prompt, user_prompt, result_text):
  """Resets the system and user prompts and the result text."""
  return gradio_system_prompt, gradio_user_prompt, ""

def clear_memory():
  """Clears the LLM memory and CUDA cache."""
  if llm is not None:
    llm.clean()
  torch.cuda.empty_cache()
  return ""

with gr.Blocks() as demo:
    with gr.Row():
      filename = gr.Textbox(label="File", lines=1, value=display_file(current_index))

    with gr.Row():
      previous_button = gr.Button("Previous")
      next_button = gr.Button("Next")

    with gr.Row():
      system_textbox = gr.Textbox(label="System Prompt", value=get_system_prompt()) #, lines=5)
      prompt_textbox = gr.Textbox(label="User Prompt", value=get_user_prompt()) #, lines=3)

    with gr.Row():
      model_dropdown = gr.Dropdown(
        label="Select MODEL",
        choices=llm.models(),
        value=DEFAULT_MODEL
      )
      with gr.Row():
        generate_button = gr.Button("Generate")
        reset_button = gr.Button("Reset")

    with gr.Row():
      result_textbox = gr.Textbox(lines=10)

    with gr.Row():
      clear_button = gr.Button("Clear memory")

    previous_button.click(go_back_file, outputs=filename)
    next_button.click(advance_file, outputs=filename)
    model_dropdown.change(fn=process_selection, inputs=model_dropdown)
    generate_button.click(generate_output, inputs=[system_textbox, prompt_textbox], outputs=result_textbox)
    reset_button.click(reset, inputs=[system_textbox, prompt_textbox, result_textbox], outputs=[system_textbox, prompt_textbox, result_textbox])
    clear_button.click(clear_memory)

demo.launch(debug=True)

In [ ]:
# Clean up

if llm is not None:
    llm.clean()
    
del llm
del demo

## Screen to Text

In [ ]:
screenshots_folder = "/home/pedro/desenvolvimento/RV_ANDROID/teste_llm/screenshots/cryptoapp"

def read_screens_info(folder_path: str):
  files = os.listdir(folder_path)
  for file in files:
    if file.endswith(".png"):
      png_file = os.path.join(folder_path, file)
      state_file = os.path.join(folder_path, file.replace(".png", ".state"))
      yield png_file, state_file

### droidbot-GPT

### rv-android

In [ ]:
# import rvandroid.parser.droidbot_state_parser as state_parser

# screen_description = state_parser.execute(state_parser.TELA_MESSAGE_DIGEST)
# print(screen_description.description)

In [ ]:
lista_imagens = []
lista_states = []
for png_file, state_file in read_screens_info(os.path.abspath(screenshots_folder)):
  lista_imagens.append(png_file)
  lista_states.append(state_file)

print(lista_states)

indice_atual = 0

def mostrar_imagem(indice):
    """Mostra a imagem atual da lista."""
    try:
        imagem = Image.open(lista_imagens[indice])
        return imagem
    except FileNotFoundError:
        return "Imagem não encontrada"

def descrever_tela():
    """Altera o texto da caixa de texto para 'Hello world'."""
    print(f"alterando texto do indice: {indice_atual}")
    
    arquivo_state = lista_states[indice_atual]
    print(f"arquivo_state={arquivo_state}")
    state_text = read_text_file(arquivo_state)
    print(f"state_text={state_text}")
    state_json = json.loads(state_text)
    print(f"state_json={state_json}")
    screen_description = state_parser.execute(state_json)
    print(f"description={screen_description.description}")
    return screen_description.description
    # return lista_states[indice_atual]

def avancar_imagem():
    """Avança para a próxima imagem da lista."""
    global indice_atual
    indice_atual = (indice_atual + 1) % len(lista_imagens)  # Volta ao início se chegar ao fim da lista
    return mostrar_imagem(indice_atual), lista_states[indice_atual]

def voltar_imagem():
    """Volta para a imagem anterior da lista."""
    global indice_atual
    indice_atual = (indice_atual - 1) % len(lista_imagens)  # Volta para o fim se chegar ao início da lista
    return mostrar_imagem(indice_atual), lista_states[indice_atual]

with gr.Blocks() as demo:
    with gr.Row():
        arquivo = gr.Textbox(value=lista_states[indice_atual], show_label=False)

    with gr.Row():
        imagem = gr.Image(value=mostrar_imagem(indice_atual), type="pil", height=500)
        texto = gr.Textbox(label="Descrição", lines=22)                

    with gr.Row():
        botao_anterior = gr.Button("Previous")
        botao_proximo = gr.Button("Next")
        botao_alterar = gr.Button("Descrever Tela")

    botao_anterior.click(voltar_imagem, outputs=[imagem, arquivo])
    botao_proximo.click(avancar_imagem, outputs=[imagem, arquivo])
    botao_alterar.click(descrever_tela, outputs=texto)

demo.launch()

### VQA

In [ ]:
def redimensionar_imagem(caminho_imagem, largura_maxima, altura_maxima):
    """
    Redimensiona uma imagem para as dimensões máximas especificadas,
    mantendo a proporção original.

    Args:
        caminho_imagem: O caminho para a imagem original.
        largura_maxima: A largura máxima desejada.
        altura_maxima: A altura máxima desejada.

    Returns:
        Uma imagem PIL redimensionada.
    """
    image = Image.open(caminho_imagem)
    image.thumbnail((largura_maxima, altura_maxima))
    return image

In [ ]:
# facebook/blip-large: Este modelo é um dos mais populares e oferece um bom equilíbrio entre desempenho e tamanho. Ele é capaz de responder a perguntas complexas sobre imagens e gerar descrições detalhadas.
# google/flan-t5-xxl: Embora seja um modelo maior, o Flan-T5-XXL pode ser usado para VQA com bom desempenho em GPUs T4, especialmente se você otimizar o uso da memória. Ele é conhecido por sua capacidade de gerar texto de alta qualidade.
# Salesforce/blip-2-flan-t5-xl: Este modelo combina o poder do BLIP-2 para visão com o modelo Flan-T5-XL para linguagem, oferecendo resultados impressionantes em tarefas de VQA.

# Escolha um modelo
# default: dandelin/vilt-b32-finetuned-vqa
# model_name = "facebook/blip-large"
# model_name = "Salesforce/blip-2-flan-t5-xl"
# model_name = "google/flan-t5-xxl"
model_name = "google/flan-t5-small"

# Crie o pipeline de VQA
vqa_pipeline = pipeline("visual-question-answering", model=model_name)

# Carregue a imagem
image_path = screenshots_folder+"/001.png"
# image_path = "/content/drive/MyDrive/llms/cryptoapp/001.png"
image = Image.open(image_path)
image = redimensionar_imagem(image_path, 512, 512)

# Defina a pergunta
# question = "O que está acontecendo na imagem?"
question = """
Instruções
Descreva a tela do aplicativo Android em detalhes, conforme as instruções fornecidas.

Formato de Resposta
A resposta deve ser estruturada em um formato de tabela ou lista, facilitando a identificação e o uso das informações para testes.

Considerações Adicionais
Adapte este prompt para suas necessidades específicas, incluindo detalhes sobre o aplicativo e os tipos de teste que você deseja realizar.
Seja claro e específico nas suas instruções para obter uma resposta mais precisa e útil.
Use a criatividade para explorar diferentes tipos de interações e ações que podem ser realizadas na tela.
"""

# Obtenha a resposta
result = vqa_pipeline(image, question)
print(result)

In [ ]:
torch.cuda.empty_cache()
del vqa_pipeline

In [ ]:
torch.cuda.empty_cache()

In [ ]:

image_path = screenshots_folder+"/002.png"

LLAVA_0_5B = "llava-hf/llava-interleave-qwen-0.5b-hf"
LLAVA_7B = "llava-hf/llava-interleave-qwen-7b-hf"

pipe = pipeline("image-text-to-text", model=LLAVA_0_5B)

messages = [
     {
         "role": "user",
         "content": [
             {
                 "type": "image",
                 "image": image_path,
             },
             {"type": "text", "text": "describe in detail the following screenshot of an android application. Identify the components (buttons, fields, spinners, etc.) that are clickable, editable or selectable, indicating possible actions on them"},
         ],
     }
 ]





In [ ]:
outputs = pipe(text=messages, max_new_tokens=300, return_full_text=True)

outputs[0]["generated_text"]

#'The screenshot displays a mobile application interface with a blue background and white text. At the top, there is a section titled "Crypto App" with a date and time stamp of 11:54. Below this, there is a section titled "Message Digest" with a dropdown menu that allows the user to select a message to digest. The selected message is highlighted with a red background.\n\nOn the right side of the screen, there is a section titled "Generate Hash" with a blue button that says "Generate Hash" and a spinning spinner icon. The spinner icon indicates that the user can generate a hash of the selected message.\n\nOn the bottom left of the screen, there is a section titled "Select Text" with a blue field that allows the user to select a text to insert into the digest. The field is highlighted with a red background.\n\nThe overall layout of the application is simple and user-friendly, with clear and concise text and icons. The design is clean and modern, with a focus on simplicity and ease of use.'}]

In [ ]:
del pipe

In [ ]:
!pip install -q transformers torch torchvision Pillow opencv-python pytesseract
!sudo apt install -y tesseract-ocr
!sudo apt install -y libtesseract-dev

from transformers import ViTFeatureExtractor, ViTModel, BertTokenizer, BertModel
from PIL import Image
import torch
import cv2
import pytesseract

# Modelos de visão
feature_extractor = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224')
model_vision = ViTModel.from_pretrained('google/vit-base-patch16-224')

# Modelos de linguagem
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model_language = BertModel.from_pretrained('bert-base-uncased')

# Configuração do Tesseract OCR
pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'  # Ajuste o caminho para o seu Tesseract



def identificar_elementos(imagem):
  """Identifica elementos interativos na imagem usando OpenCV."""
  # Converta a imagem para escala de cinza
  gray = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)

  # Use detecção de bordas para encontrar contornos
  edges = cv2.Canny(gray, 50, 150, apertureSize=3)

  # Encontre contornos
  contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

  elementos = []
  for contour in contours:
    # Obtenha as coordenadas do retângulo delimitador
    x, y, w, h = cv2.boundingRect(contour)

    # Considere apenas contornos com área razoável
    if w * h > 1000:
      elementos.append({'x': x, 'y': y, 'w': w, 'h': h})

  return elementos

def extrair_texto(imagem, elemento):
  """Extrai texto de um elemento usando Tesseract OCR."""
  x, y, w, h = elemento['x'], elemento['y'], elemento['w'], elemento['h']
  crop = imagem[y:y+h, x:x+w]
  texto = pytesseract.image_to_string(crop)
  return texto.strip()


# Carregue o screenshot
imagem = cv2.imread("/content/drive/MyDrive/llms/cryptoapp/001.png")

# Identifique os elementos interativos
elementos = identificar_elementos(imagem)

# Extraia o texto dos elementos
for elemento in elementos:
  elemento['texto'] = extrair_texto(imagem, elemento)

# Gere a descrição textual
descricao = "Tela com os seguintes elementos:\n"
for elemento in elementos:
  descricao += f"- {elemento['texto']} ({elemento['x']}, {elemento['y']}, {elemento['w']}, {elemento['h']})\n"

# Gere as possíveis ações
acoes = []
for elemento in elementos:
  if elemento['texto']:
    acoes.append(f"Interagir com o elemento: {elemento['texto']}")

print(descricao)
print(acoes)

## Summarizer

In [50]:
summarizer_system_prompt = """You are an expert in summarizing text that will serve as input for Android application screen testers. The texts contain general information about the application and you must create a summary with data that is useful for testing the software. Identifying the activities, the transitions between them and the events that activate these transitions."""
summarizer_user_prompt = "Please summarize the following texts and break it down into smaller sections.\n{}"


In [44]:
# Pipeline

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

def summarize_text(text):
  summary = summarizer(text, max_length=500, min_length=50, do_sample=False)
  return summary[0]["summary_text"]

Device set to use cuda:0


In [40]:
# text_to_summarize = f"{summarizer_system_prompt}\n{summarizer_user_prompt} \n ## Texts: \n  {response_gesda} \n {response_gator} "
text_to_summarize = f"{response_gesda}  {response_gator}"
# print(text_to_summarize)

# summary = summarize_text(text_to_summarize)
# print(summary)

In [51]:
messages = create_messages(summarizer_system_prompt, summarizer_user_prompt, f"{response_gesda}  {response_gator}")
llm = create_llm()
response = llm.generate(messages)
print(response)
llm.clean()
del llm

Based on the provided texts, I will create a summarized version with smaller sections, breaking down the data into:

**Activities**

The main activities of the application are:

* `br.unb.cic.cryptoapp.cipher.CipherActivity`: The main activity for encryption
* `br.unb.cic.cryptoapp.MainActivity`: The main entry point of the application
* `br.unb.cic.cryptoapp.messagedigest.MessageDigestActivity`: An activity for message digest calculations

**Widgets**

The widgets used in the activities are:

* `textViewCypherEncryptResult` (text view): Displays encryption results in `CipherActivity`
* `btn_cipher_encrypt` (button): Triggers encryption in `CipherActivity`
* `editTextCipherEncrypt` (edit text field): Input field for encrypting text in `CipherActivity`
* `spinnerAlgorithm` (spinner): Selects the algorithm for message digest calculations in `MessageDigestActivity`
* `editTextMessageDigest` (edit text field): Input field for hashing text in `MessageDigestActivity`
* `textViewMessageDigest

In [48]:
xxx_gesda = summarize_text(response_gesda)
print(f"gesda={xxx_gesda}")

xxx_gator = summarize_text(response_gator)
print(f"\ngator={xxx_gator}")

xxx_final = summarize_text(f"{xxx_gator}\n{xxx_gesda}")
print(f"\nfinal={xxx_final}")


gesda=Based on the provided JSON information, here is a summary of the 'cryptoapp' Android application. The application has two main activities: 'Cipher' and 'Message Digest' It also has a text view to display encryption results and a button to trigger encryption.


Your max_length is set to 500, but your input_length is only 113. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=56)



gator=This appears to be a list of events and handlers for an Android application. The format is not human-readable, but I can try to break it down into smaller sections. The menu event is handled by the `MainActivity` class, which creates a menu for the application.

final=Based on the provided JSON information, here is a summary of the 'cryptoapp' Android application. The application has two main activities: 'Cipher' and 'Message Digest' It also has a text view to display encryption results and a button to trigger encryption.


In [43]:
summarizer_model = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(summarizer_model)
print("tokenizer ...")
inputs = tokenizer(text_to_summarize, return_tensors="pt", padding=True, truncation=True, max_length=2048)
print(f"inputs ({len(inputs.input_ids)}) = {inputs}")
model = AutoModelForSeq2SeqLM.from_pretrained(summarizer_model)
outputs = model.generate(inputs.input_ids, 
                         attention_mask=inputs.attention_mask, # Use attention mask to avoid padding tokens
                         max_new_tokens=200, 
                         do_sample=False,
                         pad_token_id=tokenizer.pad_token_id) 
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Response={response}")

del model, inputs, outputs, tokenizer

tokenizer ...
inputs (1) = {'input_ids': tensor([[    0, 20930,    15,  ..., 24113,     4,     2]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]])}


IndexError: index out of range in self

## Tokenizer

In [ ]:
def create_prompt(messages: list[dict[str, str]], model=DEFAULT_MODEL):
    pass

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B', trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(LLAMA, trust_remote_code=True)

text = "I am excited to show Tokenizers in action to my LLM engineers"
tokens = tokenizer.encode(text)
tokens
tokenizer.decode(tokens)
tokenizer.batch_decode(tokens)
tokenizer.get_added_vocab()


messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]

# Quantization Config - this allows us to load the model into memory and use less memory
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

# The model
model = AutoModelForCausalLM.from_pretrained(DEFAULT_MODEL, device_map="auto", quantization_config=quant_config)

In [ ]:
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")

In [ ]:
model

In [ ]:
outputs = model.generate(inputs, max_new_tokens=80)
print(tokenizer.decode(outputs[0]))

In [ ]:
# Clean up
del inputs, outputs, model
torch.cuda.empty_cache()

In [ ]:
#

In [ ]:
# response = hf.generate(messages)
# print(response)

In [ ]:
for r in result:
    g = r["generated_text"]
    # print(g)
    for x in g:
        print(x)

xxxx

In [52]:
import boto3
import json

bedrock = boto3.client('bedrock-runtime')

body = json.dumps({
    "prompt": "\n\nHuman: Write a short story about a robot learning to feel emotions.\n\nAssistant:",
    "max_tokens_to_sample": 1000,
    "temperature": 0.5,
    "top_p": 0.9
})

model_id = "amazon.titan-tg1-xlarge" 
response = bedrock.invoke_model(
    modelId=model_id, 
    accept='*/*',
    contentType='application/json', 
    body=body
)

response_body = json.loads(response.get('body').read().decode('utf-8'))

print(response_body['completion'])

NoRegionError: You must specify a region.